In [ ]:
#Instalar Streamlit
!pip install -q streamlit

In [ ]:
import joblib
import pandas as pd

# Carregar o modelo
modelo = joblib.load('modelo_manutencao.pkl')

# Perguntar quais colunas ele exige
if hasattr(modelo, 'feature_names_in_'):
    colunas_corretas = list(modelo.feature_names_in_)
    print(colunas_corretas)
else:
    print("O modelo não salvou os nomes.")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# --- CONFIGURAÇÃO DA PÁGINA ---
st.set_page_config(page_title="Monitor Industrial AI", page_icon="🏭")

# --- CARREGAR MODELO ---
@st.cache_resource
def load_model():
    try:
        return joblib.load('modelo_manutencao.pkl')
    except:
        return None

model = load_model()

# --- TÍTULO E DESCRIÇÃO ---
st.title("🏭 Monitor de Confiabilidade Industrial")
st.markdown("""
Este sistema utiliza um modelo de **Machine Learning (Random Forest)** para prever falhas em equipamentos industriais em tempo real.
Ajuste os sensores na barra lateral para simular as condições da máquina.
""")
st.markdown("---")

# --- VERIFICAÇÃO DE SEGURANÇA ---
if model is None:
    st.error("⚠️ O arquivo do modelo ('modelo_manutencao.pkl') não foi encontrado.")
    st.info("Por favor, faça o upload do arquivo .pkl treinado para o diretório raiz.")
    st.stop()

# --- BARRA LATERAL (INPUTS) ---
st.sidebar.header("🎛️ Parâmetros Operacionais")

# Inputs Numéricos
temp_ar = st.sidebar.slider("Temperatura do Ar [K]", 290.0, 310.0, 300.0)
temp_proc = st.sidebar.slider("Temperatura do Processo [K]", 300.0, 320.0, 310.0)
rotacao = st.sidebar.slider("Rotação [rpm]", 1100, 2900, 1500)
torque = st.sidebar.slider("Torque [Nm]", 0.0, 80.0, 40.0)
desgaste = st.sidebar.slider("Desgaste da Ferramenta [min]", 0, 250, 0)

# Input Categórico (Correção importante: Selectbox)
qualidade = st.sidebar.selectbox("Qualidade da Máquina", ["Baixa (L)", "Média (M)", "Alta (H)"])

# Mapeamento da Qualidade para o formato do modelo
tipo_map = {"Baixa (L)": "L", "Média (M)": "M", "Alta (H)": "H"}
tipo_selecionado = tipo_map[qualidade]

# --- BOTÃO DE DIAGNÓSTICO ---
if st.button("🔍 Analisar Risco de Falha"):

    # 1. Preparar os dados brutos
    dados_dict = {
        'Air temperature [K]': temp_ar,
        'Process temperature [K]': temp_proc,
        'Rotational speed [rpm]': rotacao,
        'Torque [Nm]': torque,
        'Tool wear [min]': desgaste,
        'Type_M': 1 if tipo_selecionado == 'M' else 0,
        'Type_L': 1 if tipo_selecionado == 'L' else 0,
        'Type_H': 0 # Coluna dummy para consistência
    }

    # 2. Criar DataFrame
    df_input = pd.DataFrame([dados_dict])

    # 3. Reordenar colunas automaticamente (Blindagem contra erro de ordem)
    if hasattr(model, 'feature_names_in_'):
        colunas_certas = model.feature_names_in_
        df_input = df_input[colunas_certas]

    # 4. Realizar Previsão
    try:
        predicao = model.predict(df_input)[0]
        probabilidade = model.predict_proba(df_input)[0][1] # Pega a chance de falha (classe 1)

        # 5. Exibir Resultados
        col1, col2 = st.columns(2)

        with col1:
            st.metric("Probabilidade de Quebra", f"{probabilidade*100:.1f}%")

        with col2:
            if predicao == 1:
                st.error("🚨 FALHA IMINENTE DETECTADA")
                st.write("**Recomendação:** Parada imediata para manutenção.")
            else:
                st.success("✅ OPERAÇÃO NORMAL")
                st.write("**Status:** Parâmetros dentro da margem de segurança.")

    except Exception as e:
        st.error(f"Erro no processamento: {e}")

In [ ]:
# 1. Instalar biblioteca do Ngrok
!pip install pyngrok

from pyngrok import ngrok
import time

# 2. Configurar seu Token (COLE O SEU AQUI)
# Exemplo: ngrok.set_auth_token("2Fj3k4...")
ngrok.set_auth_token("INSIRA_SEU_TOKEN_AQUI")

# 3. Matar processos antigos para liberar a porta
!pkill streamlit
ngrok.kill()

# 4. Abrir o túnel na porta 8501
tunnel = ngrok.connect(8501)
print(f"🚀 SEU APP ESTÁ NO AR: {tunnel.public_url}")

# 5. Rodar o Streamlit (sem monitoramento de arquivo para evitar erro do watchdog)
!streamlit run app.py --server.fileWatcherType none

🚀 SEU APP ESTÁ NO AR: https://sawlike-deafly-terrilyn.ngrok-free.dev



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.227.6.80:8501

